# 💼 The Analyst's Notebook · Part 4
### From a report to a question

Parts 1 to 3 built a risk report: two stocks compared by hand, then a proper volatility built out of loops, then eleven instruments measured, ranked, drawn and labelled with pandas.

Every number in it describes a period that has already happened. The desk has read it, and asked the obvious next question:

> **Which of these names will be volatile next month?**

That is a different kind of question, and answering it responsibly takes more setup than most people expect. Part 4 is that setup, done properly and stopping exactly where a model would begin.

## How to work through this

- Run the **quick load** cell first. It brings back what Part 3 found and loads the price table.
- Each question builds on the last, so keep them in order and keep your variables. Later questions use the names earlier ones created.
- Cells with `...` are blanks. The notebook runs cleanly even before you fill them in, so **Run all** is always safe.
- Hints and solutions are folded under each question. Work first, then check.

**You will not fit a model here.** You will define the problem so precisely that fitting one becomes the easy part, which is the honest order to do it in.

*Stuck for more than 15 minutes? Ask a friend, ask an AI for a hint (not the answer), or email me at `jobo@econ.au.dk`.*

---

## ⚙️ Quick load

The packages, the price table, and what Part 3 left you. Run it and read what it prints.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def data_path(filename):
    """Where the course CSV files are, wherever you happen to be running."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if REPO_RAW_URL is not None:
        return REPO_RAW_URL + filename
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


# The whole universe: eleven instruments, 2015 to 2024
prices = pd.read_csv(data_path("prices.csv"), parse_dates=["date"])
wide = prices.pivot(index="date", columns="ticker", values="close")
rets = wide.pct_change()
TICKERS = sorted(prices["ticker"].unique())

# --- What Part 3 found: annualised volatility in 2024 ---
part3_annual_vol = {
    "NVDA": 0.525,
    "DIS": 0.258,
    "JPM": 0.235,
    "AAPL": 0.227,
    "MSFT": 0.200,
    "XOM": 0.192,
    "WMT": 0.177,
    "JNJ": 0.151,
    "PG": 0.150,
    "KO": 0.128,
    "SPY": 0.126,
}
part3_riskiest = "NVDA"
part3_calmest = "SPY"

print("Loaded prices:", prices.shape[0], "rows")
print("Instruments  :", ", ".join(TICKERS))
print()
print("Part 3 recap: annualised volatility in 2024")
for ticker, value in part3_annual_vol.items():
    print(f"  {ticker:5} {value:.3f}")
print(f"  riskiest: {part3_riskiest}   calmest: {part3_calmest}")
print()
print("Part 3 also found that the ranking only partly carried from 2015-2022")
print("into 2023-2024. That gap is what Part 4 is about.")

---

### Q1 · Say exactly what you are predicting

"Will it be volatile next month?" is not yet a question a computer can answer. Pin it down.

The target is **the volatility of the next twenty trading days**: the standard deviation of the daily returns from tomorrow to twenty days from now. Build it for Apple, as a Series called `target`.

$$\text{target}_t = \text{sd}\big(r_{t+1},\, r_{t+2},\, \ldots,\, r_{t+20}\big)$$

In [ ]:
target = ...
target

<details>
<summary>💡 Hint 1</summary>

A rolling standard deviation is `rets['AAPL'].rolling(20).std()`. That is the volatility of the twenty days **ending** today.

</details>

<details>
<summary>💡 Hint 2</summary>

To move it so that each row holds the *next* twenty days instead, shift it backwards: `.shift(-20)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
target = rets['AAPL'].rolling(20).std().shift(-20)
target
```

One number per day, and every one of them describes a period that had not happened yet on that date. That is exactly what makes it a target rather than a feature.

*A note on why volatility and not return: returns are close to unpredictable, and a case study that ends in `R2 = -0.02` teaches you less. Volatility clusters, so there is something real to find.*

</details>

---

### Q2 · Only what you would have known

Now the features. Each one must be computable **on the day itself**, using nothing from the future. Build three, each over the last twenty trading days:

- `vol_20d`: the volatility of the returns up to and including today
- `ret_20d`: the average daily return over the same window
- `up_20d`: the share of those days that were up

Collect them into a DataFrame called `features`, with exactly those three column names.

In [ ]:
features = ...
features

<details>
<summary>💡 Hint 1</summary>

The first two are `rets['AAPL'].rolling(20).std()` and `rets['AAPL'].rolling(20).mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

For the third, `(rets['AAPL'] > 0)` gives True and False, and a rolling mean of those is the share that were True.

</details>

<details>
<summary>✅ Solution</summary>

```python
features = pd.DataFrame({
    'vol_20d': rets['AAPL'].rolling(20).std(),
    'ret_20d': rets['AAPL'].rolling(20).mean(),
    'up_20d': (rets['AAPL'] > 0).rolling(20).mean(),
})
features
```

Note the asymmetry: the features look **backwards** twenty days, the target looks **forwards** twenty days, and they never overlap. Draw it on paper if it helps. Getting this wrong is the most expensive mistake in applied finance, and it is invisible in the output.

</details>

---

### Q3 · The feature that knew too much

A colleague built the table below while you were in the meeting. It has one feature and the same target, and they are pleased with it, because the two columns line up almost perfectly.

Run it, read the two `shift` calls carefully, and work out why nobody could ever have used this table. Then repair the feature.

In [ ]:
leaky = pd.DataFrame({
    'vol_feature': rets['AAPL'].rolling(20).std().shift(-20),
    'vol_next': rets['AAPL'].rolling(20).std().shift(-20),
}).dropna()
print('correlation:', leaky['vol_feature'].corr(leaky['vol_next']))

honest = ...
print('correlation:', ...)

<details>
<summary>💡 Hint 1</summary>

Look at the two shifts. They are identical, so the feature **is** the target: it is the volatility of the next twenty days, which nobody knows yet.

</details>

<details>
<summary>💡 Hint 2</summary>

A legitimate feature looks backwards. Drop the shift from the feature and leave it on the target only.

</details>

<details>
<summary>✅ Solution</summary>

```python
leaky = pd.DataFrame({
    'vol_feature': rets['AAPL'].rolling(20).std().shift(-20),
    'vol_next': rets['AAPL'].rolling(20).std().shift(-20),
}).dropna()
print('correlation:', leaky['vol_feature'].corr(leaky['vol_next']))

honest = pd.DataFrame({
    'vol_feature': rets['AAPL'].rolling(20).std(),
    'vol_next': rets['AAPL'].rolling(20).std().shift(-20),
}).dropna()
print('correlation:', honest['vol_feature'].corr(honest['vol_next']))
```

A correlation of **1.000** against **0.50** once repaired.

A perfect correlation between a feature and a target is never good news. It means the feature contains the answer, and the only question left is how it got there. Here it is obvious because both shifts are on the same line; in a real project the leak arrives through a join with a table someone else built, and the only symptom is a result that is too good.

**Treat a suspiciously strong feature as a bug report, not a discovery.**

</details>

---

### Q4 · One table, and its shape

Put the features and the target into one table called `table`, drop the rows that are not complete, and report `n` and `p`.

In [ ]:
table = ...
n = ...
p = ...
print('n =', n)
print('p =', p)

<details>
<summary>💡 Hint 1</summary>

`features['vol_next'] = target` adds the target as a fourth column, then `.dropna()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`table.shape` gives `(rows, columns)`, and one of those columns is the target, so `p` is one less than the column count.

</details>

<details>
<summary>✅ Solution</summary>

```python
features['vol_next'] = target
table = features.dropna()

n = table.shape[0]
p = table.shape[1] - 1
print('n =', n)
print('p =', p)
```

**n = 2,476 and p = 3.** This is the object the whole of Session 4 was about: rows are observations, three columns are features, one is the target.

Everything you do from here is either about improving those columns or about finding an honest way to score a rule that maps them to the last one.

</details>

---

### Q5 · What the cleaning cost

Count how many rows the `dropna()` threw away, and work out where they were.

In [ ]:
n_before = ...
n_after = ...
print('rows before:', n_before)
print('rows after :', n_after)
print('lost       :', ...)

<details>
<summary>💡 Hint 1</summary>

`features` before dropping still has every date, so `len(features)` is the before count.

</details>

<details>
<summary>💡 Hint 2</summary>

The lost rows are the difference. Think about which end of the sample each kind of missing row sits at.

</details>

<details>
<summary>✅ Solution</summary>

```python
n_before = len(features)
n_after = len(table)
print('rows before:', n_before)
print('rows after :', n_after)
print('lost       :', n_before - n_after)
```

**40 rows lost from 2,516.** Nineteen at the start, where the twenty-day windows had not filled up yet, and twenty at the end, where the target reaches past the last date in the file.

Neither loss is a data quality problem. Both are the price of asking a question about a window of time, and both are worth stating out loud rather than discovering later.

</details>

---

### Q6 · The extreme months

Find the three dates with the **largest** target values, and say what happened.

In [ ]:
worst = ...
worst

<details>
<summary>💡 Hint 1</summary>

`table['vol_next'].sort_values(ascending=False)` puts the biggest first.

</details>

<details>
<summary>💡 Hint 2</summary>

Then take the top three with `.head(3)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
worst = table['vol_next'].sort_values(ascending=False).head(3)
worst
```

All three are late February 2020, with the target reaching **0.0680** on 28 February 2020. Standing on those dates, the next twenty trading days were the fastest crash in modern market history.

So: do you drop them? **No.** A risk model that has never seen a crash is not a risk model. This is the Session 4 point about outliers in finance, arriving in your own data: the extremes are not contamination, they are the subject.

</details>

---

### Q7 · The shape of the target

Compare the **mean** and the **median** of the target, and say what the gap tells you about its distribution.

In [ ]:
target_mean = ...
target_median = ...
print('mean  :', target_mean)
print('median:', target_median)
print('ratio :', ...)

<details>
<summary>💡 Hint</summary>

Both are methods on `table['vol_next']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
target_mean = table['vol_next'].mean()
target_median = table['vol_next'].median()
print('mean  :', target_mean)
print('median:', target_median)
print('ratio :', target_mean / target_median)
```

Mean **0.01625**, median **0.01466**. The mean sits above the median, so the distribution has a long right tail: most months are calm and a few are terrible.

That has a direct consequence for Part 4's scoring. A squared metric will be dominated by those few months, and an absolute one will not. Neither is wrong. You just have to decide, in advance, which question you are asking.

</details>

---

### Q8 · Split, and say why it has to be a date

Split `table` into `train` (everything up to the end of 2022) and `test` (2023 onwards), and report the size of each.

In [ ]:
train = ...
test = ...
print('train rows:', ...)
print('test rows :', ...)

<details>
<summary>💡 Hint</summary>

The index is dates, so `table.loc[:'2022-12-31']` and `table.loc['2023-01-01':]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
train = table.loc[:'2022-12-31']
test = table.loc['2023-01-01':]
print('train rows:', len(train))
print('test rows :', len(test))
```

**1,994 training rows and 482 test rows.**

It has to be a date rather than a random draw of rows. A randomly chosen test day would sit between two training days, so anything you built would already know roughly what the market was doing that week. The test set has to be the *future*, because that is the only situation you will ever actually be in.

</details>

---

### Q9 · The baseline nobody is allowed to skip

Before any model, score the laziest rule there is: predict the **average of the training target** on every test day. Report its RMSE and MAE.

$$\text{RMSE}=\sqrt{\tfrac{1}{n}\sum_i (y_i-\hat{y}_i)^2}, \qquad \text{MAE}=\tfrac{1}{n}\sum_i |y_i-\hat{y}_i|$$

In [ ]:
base_pred = ...
base_resid = ...

base_rmse = ...
base_mae = ...
print('baseline RMSE:', base_rmse)
print('baseline MAE :', base_mae)

<details>
<summary>💡 Hint 1</summary>

The prediction is one number repeated: `train['vol_next'].mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`base_resid = test['vol_next'] - base_pred`, then RMSE is `np.sqrt((base_resid ** 2).mean())` and MAE is `base_resid.abs().mean()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
base_pred = train['vol_next'].mean()
base_resid = test['vol_next'] - base_pred

base_rmse = np.sqrt((base_resid ** 2).mean())
base_mae = base_resid.abs().mean()
print('baseline RMSE:', base_rmse)
print('baseline MAE :', base_mae)
```

RMSE **0.00514**, MAE **0.00435**. Remember these two numbers. Anything you build later has to beat them, and a surprising amount of published work never checks.

</details>

---

### Q10 · The rule a trader would actually use

Now a real rule, and still not a model: **next month will look like this month**. Predict each test day's target with that day's `vol_20d`, and score it the same two ways.

In [ ]:
pers_pred = ...
pers_resid = ...

pers_rmse = ...
pers_mae = ...
print('persistence RMSE:', pers_rmse)
print('persistence MAE :', pers_mae)

<details>
<summary>💡 Hint 1</summary>

The prediction is simply the `vol_20d` column of `test`.

</details>

<details>
<summary>💡 Hint 2</summary>

Everything else is exactly as in Q9.

</details>

<details>
<summary>✅ Solution</summary>

```python
pers_pred = test['vol_20d']
pers_resid = test['vol_next'] - pers_pred

pers_rmse = np.sqrt((pers_resid ** 2).mean())
pers_mae = pers_resid.abs().mean()
print('persistence RMSE:', pers_rmse)
print('persistence MAE :', pers_mae)
```

RMSE **0.00417** against the baseline's 0.00514, and MAE **0.00335** against 0.00435. A rule with no parameters, no fitting and no data science beats the average on both.

That is volatility clustering, and it is why forecasting volatility is a real business while forecasting returns is mostly not.

</details>

---

### Q11 · How much better, in one number

Express that improvement as an $R^2$: how much of the test variation the persistence rule accounts for, measured against the training-mean baseline.

$$R^2 = 1 - \frac{\sum_i (y_i-\hat{y}_i)^2}{\sum_i (y_i-\bar{y}_{\text{train}})^2}$$

In [ ]:
rss = ...
tss = ...
pers_r2 = ...
print('persistence R2 on the test years:', pers_r2)

<details>
<summary>💡 Hint 1</summary>

`rss` is the sum of squared persistence residuals: `(pers_resid ** 2).sum()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`tss` uses the **training** mean as the baseline: `((test['vol_next'] - train['vol_next'].mean()) ** 2).sum()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
rss = (pers_resid ** 2).sum()
tss = ((test['vol_next'] - train['vol_next'].mean()) ** 2).sum()
pers_r2 = 1 - rss / tss
print('persistence R2 on the test years:', pers_r2)
```

**0.340.** On data it had never seen, a one-line rule accounts for about a third of the variation in next month's volatility.

Hold on to that number too. It is now the bar. A model that takes three features and a fitting procedure and lands below it has earned nothing.

</details>

---

### Q12 · Draw the forecast against what happened

Two numbers told you the persistence rule beats the baseline. A picture tells you *how*.

On the test years only, plot the realised target and the persistence prediction on the same axes, and label both.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
...
ax.legend()
plt.show()

<details>
<summary>💡 Hint 1</summary>

The two series are `test['vol_next']` and `test['vol_20d']`, both against `test.index`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.plot(test.index, test['vol_next'], label='what happened')` and the same for the prediction, then a `set_ylabel` and a `set_title`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(test.index, test['vol_next'], linewidth=1.6, label='what happened')
ax.plot(test.index, test['vol_20d'], linewidth=1.4, label='persistence forecast')
ax.set_ylabel('20-day volatility')
ax.set_title('Apple, 2023 to 2024: forecast against outcome', loc='left')
ax.legend()
plt.show()
```

The two lines have the same shape, and the forecast is the outcome **shifted to the right**. That is what a persistence rule is: it repeats what just happened, so it is always a month late to every turn.

Look at where the gaps are widest. They are at the turning points, exactly where a forecast would have been worth having. A rule that is right in the calm stretches and wrong at every turn can still post a respectable R2, which is why you look at the picture as well as the number.

</details>

---

### Q13 · Does it hold for the whole desk?

Apple is one name. Repeat the whole thing for all eleven: build the table, split it at the same date, and compute the persistence $R^2$. Collect the answers in a Series, worst last.

In [ ]:
r2_by_ticker = {}

for ticker in TICKERS:
    t = pd.DataFrame({'vol_20d': rets[ticker].rolling(20).std()})
    t['vol_next'] = ...
    t = t.dropna()

    tr = ...
    te = ...

    r2_by_ticker[ticker] = ...

result = ...
result

<details>
<summary>💡 Hint 1</summary>

Inside the loop, the target is the same construction as Q1 with `ticker` in place of `'AAPL'`, and the split is the same two `.loc` slices as Q8.

</details>

<details>
<summary>💡 Hint 2</summary>

The R2 is the Q11 expression: `1 - ((te['vol_next'] - te['vol_20d']) ** 2).sum() / ((te['vol_next'] - tr['vol_next'].mean()) ** 2).sum()`. Afterwards, `pd.Series(r2_by_ticker).sort_values(ascending=False)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
r2_by_ticker = {}

for ticker in TICKERS:
    t = pd.DataFrame({'vol_20d': rets[ticker].rolling(20).std()})
    t['vol_next'] = rets[ticker].rolling(20).std().shift(-20)
    t = t.dropna()

    tr = t.loc[:'2022-12-31']
    te = t.loc['2023-01-01':]

    rss = ((te['vol_next'] - te['vol_20d']) ** 2).sum()
    tss = ((te['vol_next'] - tr['vol_next'].mean()) ** 2).sum()
    r2_by_ticker[ticker] = 1 - rss / tss

result = pd.Series(r2_by_ticker).sort_values(ascending=False)
result
```

**Positive for only 4 of the eleven.** AAPL leads at 0.34; DIS is at -1.58, which means the rule is far worse there than simply predicting the training average.

This is the most useful result in the notebook. A rule that looked convincing on one name falls apart across the desk, which is precisely the gap a real model is supposed to close. It also shows why you evaluate per instrument and not on a single flattering example.

</details>

---

### Q14 · Does Part 3's ranking tell you anything?

The quick load restated Part 3's finding: Nvidia was the most volatile name of 2024, the index fund the calmest. A natural guess is that the calm names are the predictable ones.

You now have the evidence to check it. Put the 2024 volatilities beside the R2 values from Q13 and measure how strongly they move together.

In [ ]:
vol_2024 = pd.Series(part3_annual_vol)

comparison = pd.DataFrame({'vol_2024': vol_2024, 'r2': ...})
correlation = ...

print(comparison.sort_values('vol_2024', ascending=False).round(3))
print('correlation:', correlation)

<details>
<summary>💡 Hint 1</summary>

`r2_by_ticker` from Q13 is a dictionary, so `pd.Series(r2_by_ticker)` lines it up against the volatilities by ticker.

</details>

<details>
<summary>💡 Hint 2</summary>

`comparison['vol_2024'].corr(comparison['r2'])` measures how they move together, between -1 and +1.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol_2024 = pd.Series(part3_annual_vol)

comparison = pd.DataFrame({'vol_2024': vol_2024, 'r2': pd.Series(r2_by_ticker)})
correlation = comparison['vol_2024'].corr(comparison['r2'])

print(comparison.sort_values('vol_2024', ascending=False).round(3))
print('correlation:', correlation)
```

**-0.35.** Weakly negative, which says the more volatile names were, if anything, slightly harder to forecast.

Now be careful, because this is where a report goes wrong. You have **eleven points**. A correlation of -0.35 on eleven points is roughly what you get from pure noise, and the two most extreme names, Nvidia and Disney, are doing most of the work. The honest sentence is *this sample cannot tell*, not *volatile names are harder to predict*.

Part 3's ranking was a good description of 2024. It is not a guide to which names a model will do well on, and you now know that because you checked rather than assumed.

</details>

---

### Q15 · The same question, asked as a label

The desk does not really need a number. It needs to know **which names to watch**. Turn the problem into a classification: a month counts as *high volatility* if the target is above the **median of the training target**.

Compute that threshold, then build `high_true` (it really was high) and `high_pred` (this month was high, so the rule says next month will be).

In [ ]:
threshold = ...

high_true = ...
high_pred = ...

print('threshold      :', threshold)
print('share high, test:', ...)

<details>
<summary>💡 Hint 1</summary>

The threshold is `train['vol_next'].median()`, and it must come from the training rows only.

</details>

<details>
<summary>💡 Hint 2</summary>

`high_true = test['vol_next'] > threshold` and `high_pred = test['vol_20d'] > threshold`.

</details>

<details>
<summary>✅ Solution</summary>

```python
threshold = train['vol_next'].median()

high_true = test['vol_next'] > threshold
high_pred = test['vol_20d'] > threshold

print('threshold      :', threshold)
print('share high, test:', high_true.mean())
```

The threshold is **0.01540**, and only **25.1%** of the test months clear it.

Read that carefully. By construction half of the *training* months were above the training median, so the quiet market of 2023 and 2024 has left the two periods badly out of balance. A threshold learned on one regime does not transfer to another, and no amount of modelling fixes that on its own.

</details>

---

### Q16 · Score the warning system

Count the four outcomes for that rule, then compute precision and recall.

$$\text{precision}=\frac{TP}{TP+FP}, \qquad \text{recall}=\frac{TP}{TP+FN}$$

In [ ]:
tp = ...
fp = ...
fn = ...
tn = ...

precision = ...
recall = ...
print('TP', tp, ' FP', fp, ' FN', fn, ' TN', tn)
print('precision:', precision)
print('recall   :', recall)

<details>
<summary>💡 Hint 1</summary>

Combine the two boolean Series with `&`, and flip one with `~`: `(high_pred & high_true).sum()` is TP.

</details>

<details>
<summary>💡 Hint 2</summary>

Precision is `tp / (tp + fp)` and recall is `tp / (tp + fn)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
tp = (high_pred & high_true).sum()
fp = (high_pred & ~high_true).sum()
fn = (~high_pred & high_true).sum()
tn = (~high_pred & ~high_true).sum()

precision = tp / (tp + fp)
recall = tp / (tp + fn)
print('TP', tp, ' FP', fp, ' FN', fn, ' TN', tn)
print('precision:', precision)
print('recall   :', recall)
```

TP **55**, FP **86**, FN **66**, TN **275**. Precision **0.390** and recall **0.455**, on an accuracy of 0.685.

So the warning system fires often and is right about four times in ten, and it still misses more than half of the months that really were turbulent. Which of those two failures is worse is not a statistical question. On a risk desk a missed turbulent month costs far more than a false alarm, so you would move the threshold down and accept the extra noise.

</details>

---

### Q17 · Write the specification

You now have everything a modeller would need. Write it down in one place, so somebody else could pick it up and be judged fairly.

Fill in the dictionary, then write `describe(spec)`: a function that prints it line by line and ends with a one-sentence verdict on whether the bar is worth clearing.

A dictionary, a loop, an f-string and an `if`. Everything in this last cell of the case comes from Sessions 1 and 2.

In [ ]:
spec = {
    'target': ...,
    'features': ...,
    'n_train': ...,
    'n_test': ...,
    'split': ...,
    'metric': ...,
    'baseline_rmse': ...,
    'bar_to_beat_r2': ...,
}

def describe(spec):
    ...

describe(spec)

<details>
<summary>💡 Hint 1</summary>

Most of the values are already sitting in variables you created: `base_rmse` from Q9 and `pers_r2` from Q11.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the function, `for key in spec:` then `print(f"{key:15} {spec[key]}")`, exactly as in Session 2's dictionary loop.

</details>

<details>
<summary>💡 Hint 3</summary>

For the verdict, an `if` on `spec['bar_to_beat_r2']`: above about 0.2 there is real signal to beat, below it the naive rule is barely better than nothing.

</details>

<details>
<summary>✅ Solution</summary>

```python
spec = {
    'target': 'sd of daily returns over the next 20 trading days',
    'features': list(train.columns.drop('vol_next')),
    'n_train': len(train),
    'n_test': len(test),
    'split': 'by date: train to 2022-12-31, test from 2023-01-01',
    'metric': 'RMSE, with MAE reported alongside because the target is skewed',
    'baseline_rmse': round(base_rmse, 5),
    'bar_to_beat_r2': round(pers_r2, 3),
}

def describe(spec):
    """Print a learning problem, and say whether the bar is a serious one."""
    for key in spec:
        print(f"{key:15} {spec[key]}")

    bar = spec['bar_to_beat_r2']
    if bar > 0.2:
        print(f"\nA model must beat R2 = {bar:.2f}. That is a real bar, not a formality.")
    else:
        print(f"\nThe naive rule only reaches R2 = {bar:.2f}, so almost anything should beat it.")

describe(spec)
```

That is a machine learning problem, fully specified, and not one model has been fitted.

Look at what the last cell of this case is made of: a dictionary and a loop from Session 2, an f-string from Session 1, a function with a docstring from Session 2, and numbers from Session 4. Four weeks of tools in fifteen lines, which is what the exam will ask of you.

Everything in the rest of the course slots into the one gap left in it: the rule that turns those three feature columns into a prediction. Choosing that rule well is what the next block is about, and you now know exactly what it has to beat.

</details>

---

## 🧭 What you have now

| what | where it lives |
|:--|:--|
| the question, written as a target | `target` |
| features that cannot see the future | `features` |
| the learning table, complete rows only | `table`, `n`, `p` |
| an honest split by date | `train`, `test` |
| the baseline nobody may skip | `base_rmse`, `base_mae` |
| the bar a model has to clear | `pers_rmse`, `pers_r2` |
| the same problem as a classification | `threshold`, `precision`, `recall` |
| the whole thing, written down | `spec` |


## What is still wrong with it

Three things, and noticing them yourself is the point of having built it:

- **The rows are not independent.** Consecutive days share nineteen of their twenty return observations, so 2,457 rows is nowhere near 2,457 independent pieces of evidence. Any confidence interval you computed here would be far too narrow.
- **One split is one experiment.** You chose the end of 2022 because it was convenient. A different cut could easily give a different verdict.
- **The regimes differ.** The training years contain a crash and the test years do not, which is why the classification threshold transferred so badly.

All three have names and all three have answers, and they are the subject of the next block: model selection, cross-validation, and the discipline of not believing your own first result.

## The end of the beginning

Four sessions ago you compared two stocks with a subtraction. You now have a fully specified prediction problem on eleven instruments, with a target, honest features, a dated split, two baselines and a metric you can defend.

The one thing you have never done is fit a model. That was deliberate. Fitting is the easy part, it is three lines of scikit-learn, and doing it before the work above is how people produce results that cannot survive contact with next month.

**Next block:** the three lines, and everything that has to be true before you are allowed to believe them.